In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. LOAD INTEGRATED BASE

# Corrected ingestion method from original to_parquet execution error
df = pd.read_parquet('../data/03_integrated/logistics_base.parquet')

In [ ]:
# Coerce dates using mixed format parsing to handle timezone/formatting variance safely
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', dayfirst=True)
df['ship_date'] = pd.to_datetime(df['ship_date'], format='mixed', dayfirst=True)

In [ ]:
# 2. FORENSIC TEMPORAL DECRYPTION

# Calculate raw duration to identify artificial programmatic shifts in the data
df['raw_lead_time_days'] = (df['ship_date'] - df['order_date']).dt.days

In [ ]:
def reverse_temporal_mangle(raw_days):
    """
    Forensic function to reverse programmatic year-over-year shifts.
    Identified Offsets: 904 days (2.5yr), 1269 days (3.5yr), 1634 days (4.5yr).
    """
    if raw_days < 1100:
        return raw_days - 900
    elif raw_days < 1450:
        return raw_days - 1269
    else :
        return raw_days - 1634

# Apply temporal decryption to establish the true target variable
df['shipping_lead_time'] = df['raw_lead_time_days'].apply(reverse_temporal_mangle)
df['ship_date'] = df['order_date'] + pd.to_timedelta(df['shipping_lead_time'], unit='D')

In [ ]:
# 3. KPI ENGINEERING: SLA & DELAYS

# Define allowable transit limits by service tier
sla_mapping = {
    'same_day': 0,
    'first_class': 2,
    'second_class': 3,
    'standard_class': 6
}

In [ ]:
# Standardize qualitative SLA keys to map quantitative targets
df['ship_mode_key'] = df['ship_mode'].str.lower().str.replace(' ', '_')
df['sla_target'] = df['ship_mode_key'].map(sla_mapping)

In [ ]:
# Generate critical binary indicator for on-time delivery tracking
df['is_delayed'] = df['shipping_lead_time'] > df['sla_target']

In [ ]:
# Quantify delay severity (days past SLA); floor early shipments at 0 to prevent skew
df['delay_magnitude'] = (df['shipping_lead_time'] - df['sla_target']).clip(lower=0)

In [ ]:
# 4. ROUTE INTELLIGENCE FEATURES

# Construct composite origin-destination variables for geospatial network analysis
df['route_state'] = df['factory'] + " -> " + df['state_province']
df['route_region'] = df['factory'] + " -> " + df['region']

In [ ]:
# Financial Efficiency Metric
df['gross_margin_pct'] = (df['gross_profit'] / df['sales']).round(4)

In [ ]:
# 5. DATA QUALITY & VALIDATION SUMMARY
print("--- LOGISTICS INTELLIGENCE SUMMARY ---")
print(f"Total Shipments Analyzed: {len(df):,}")
print(f"Operational Lead Time Range: {df['shipping_lead_time'].min()} to {df['shipping_lead_time'].max()} days")
print(f"Mean Shipping Lead Time: {df['shipping_lead_time'].mean():.2f} days")
print(f"Overall Network Delay Rate: {df['is_delayed'].mean():.2%}")

In [ ]:
# Ensure structural integrity of the decrypted timelines
assert df['shipping_lead_time'].min() >= 0, "Error: Negative lead times detected after decryption!"

In [ ]:
# 6. EXPORT TO ENGINEERED LAYER

# Strip intermediate transformations to optimize dataset for analytical querying
cols_to_drop = ['raw_lead_time_days', 'ship_mode_key']
df_final = df.drop(columns=cols_to_drop)

In [ ]:
df_final.to_parquet('../data/04_engineered/shipment_features.parquet', index=False)
print("\nSUCCESS: Engineered dataset exported to 'data/04_engineered/shipment_features.csv'")